In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import google.generativeai as genai
import gradio as gr
import warnings
warnings.filterwarnings('ignore')
 
# Load BioBERT Model
 

print("\n Loading BioBERT model...")
MODEL_PATH = './model'

biobert_tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
biobert_model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
biobert_model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
biobert_model.to(device)
print(f" BioBERT loaded on {device.upper()}")

 
# Setup Gemini API
 

print("\n Setting up Gemini API...")
GEMINI_API_KEY = "YOUROWNAPIKEY"

genai.configure(api_key=GEMINI_API_KEY)
gemini_model = genai.GenerativeModel('models/gemini-2.5-flash')
print("Gemini loaded")

 
# BioBERT Prediction Function
 

def predict_trial_outcome(title, description, eligibility=""):
    """Predict clinical trial outcome using BioBERT."""
    full_text = f"{title} [SEP] {description} [SEP] {eligibility}"
    
    inputs = biobert_tokenizer(
        full_text,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        padding=True
    )
    
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = biobert_model(**inputs)  # Uses biobert_model, not model
        logits = outputs.logits
        probs = torch.nn.functional.softmax(logits, dim=1)
    
    failure_prob = probs[0][0].item()
    success_prob = probs[0][1].item()
    
    prediction = "SUCCESS" if success_prob > 0.5 else "FAILURE"
    confidence = max(success_prob, failure_prob)
    
    if confidence > 0.8:
        confidence_level = "High"
    elif confidence > 0.6:
        confidence_level = "Medium"
    else:
        confidence_level = "Low"
    
    risk_factors = []
    desc_lower = description.lower()
    
    if "pilot" in desc_lower or "feasibility" in desc_lower:
        risk_factors.append("Exploratory/pilot study design")
    if "phase 1" in desc_lower or "phase i" in desc_lower:
        risk_factors.append("Early phase trial")
    if not any(word in desc_lower for word in ["randomized", "controlled", "blind"]):
        risk_factors.append("Non-randomized design")
    if "rare" in desc_lower or "orphan" in desc_lower:
        risk_factors.append("Rare disease - recruitment challenges")
    
    positive_factors = []
    if "randomized" in desc_lower and "controlled" in desc_lower:
        positive_factors.append("Randomized controlled design")
    if "double-blind" in desc_lower or "double blind" in desc_lower:
        positive_factors.append("Double-blind methodology")
    if "placebo" in desc_lower:
        positive_factors.append("Placebo-controlled")
    
    return {
        'prediction': prediction,
        'success_probability': round(success_prob * 100, 1),
        'failure_probability': round(failure_prob * 100, 1),
        'confidence': confidence_level,
        'confidence_score': round(confidence * 100, 1),
        'risk_factors': risk_factors if risk_factors else ["No major risk factors identified"],
        'positive_factors': positive_factors if positive_factors else ["Standard trial design"]
    }

 
# Gemini Explanation Function
 

def explain_prediction(prediction_results, trial_title, trial_description, user_question=None):
    """Use Gemini to explain the BioBERT prediction."""
    if user_question is None:
        prompt = f"""You are a medical AI assistant helping doctors understand clinical trial predictions.

A machine learning model (BioBERT fine-tuned on 2000+ clinical trials) has analyzed this trial:

**Trial Title:** {trial_title}
**Trial Description:** {trial_description}

**Model's Prediction:**
- Outcome: {prediction_results['prediction']}
- Success Probability: {prediction_results['success_probability']}%
- Confidence: {prediction_results['confidence']}

**Risk Factors:** {', '.join(prediction_results['risk_factors'])}
**Positive Factors:** {', '.join(prediction_results['positive_factors'])}

Provide a clear 3-4 sentence explanation of why the model made this prediction and what it means for researchers."""
    else:
        prompt = f"""You are a medical AI assistant.

A BioBERT model predicted: {prediction_results['prediction']} ({prediction_results['success_probability']}% success)
Trial: {trial_title}
Question: {user_question}

Provide a clear, concise answer (2-3 sentences)."""

    try:
        response = gemini_model.generate_content(prompt)  # Uses gemini_model
        return response.text
    except Exception as e:
        return f"Error: {e}"

 
# Gradio Interface Functions
 

def analyze_trial(title, description, eligibility=""):
    """Main analysis function."""
    if not title or not description:
        return " Please provide at least a title and description.", "", ""
    
    try:
        prediction = predict_trial_outcome(title, description, eligibility)
        explanation = explain_prediction(prediction, title, description)
        
        results_text = f"""
# Prediction Results

## Outcome: **{prediction['prediction']}**

### Probabilities:
-  **Success**: {prediction['success_probability']}%
-  **Failure**: {prediction['failure_probability']}%
- **Confidence**: {prediction['confidence']} ({prediction['confidence_score']}%)

---

### Positive Factors:
{chr(10).join('✓ ' + factor for factor in prediction['positive_factors'])}

### Risk Factors:
{chr(10).join('⚠️ ' + factor for factor in prediction['risk_factors'])}
"""
        return results_text, explanation, prediction
    except Exception as e:
        return f" Error: {e}", "", None

def answer_question(question, trial_title, trial_description, current_prediction):
    """Answer follow-up questions."""
    if not question:
        return "Please ask a question."
    if current_prediction is None:
        return " Please analyze a trial first."
    
    try:
        answer = explain_prediction(current_prediction, trial_title, trial_description, user_question=question)
        return f"**Q:** {question}\n\n**A:** {answer}"
    except Exception as e:
        return f"Error: {e}"

print("\n All functions loaded!")
print("="*70)
print("Ready to launch Gradio interface!")

In [ ]:
# Custom CSS
custom_css = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&display=swap');

* {
    font-family: 'Inter', -apple-system, BlinkMacSystemFont, sans-serif;
}

.gradio-container {
    max-width: 1400px !important;
    margin: auto;
}

/* Header styling */
.main-header {
    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
    padding: 40px;
    border-radius: 16px;
    text-align: center;
    color: white;
    margin-bottom: 30px;
    box-shadow: 0 10px 40px rgba(102, 126, 234, 0.3);
}

.main-header h1 {
    font-size: 42px;
    font-weight: 700;
    margin-bottom: 10px;
    text-shadow: 0 2px 10px rgba(0,0,0,0.2);
}

.main-header p {
    font-size: 18px;
    opacity: 0.95;
    margin: 5px 0;
}

/* Info box styling */
.info-box {
    background: linear-gradient(135deg, #f5f7fa 0%, #c3cfe2 100%);
    border-left: 5px solid #667eea;
    padding: 20px;
    border-radius: 12px;
    margin: 15px 0;
    box-shadow: 0 4px 15px rgba(0,0,0,0.1);
}

/* Section headers */
.section-header {
    background: linear-gradient(90deg, #667eea 0%, #764ba2 100%);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    font-size: 24px;
    font-weight: 700;
    margin: 20px 0 15px 0;
}

/* Input fields */
.input-field {
    border: 2px solid #e0e7ff;
    border-radius: 10px;
    transition: all 0.3s ease;
}

.input-field:focus {
    border-color: #667eea;
    box-shadow: 0 0 0 3px rgba(102, 126, 234, 0.1);
}

/* Buttons */
.primary-button {
    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important;
    border: none !important;
    color: white !important;
    font-weight: 600 !important;
    padding: 12px 32px !important;
    border-radius: 10px !important;
    box-shadow: 0 4px 15px rgba(102, 126, 234, 0.4) !important;
    transition: all 0.3s ease !important;
}

.primary-button:hover {
    transform: translateY(-2px) !important;
    box-shadow: 0 6px 20px rgba(102, 126, 234, 0.5) !important;
}

/* Results box */
.results-box {
    background: white;
    border-radius: 12px;
    padding: 25px;
    box-shadow: 0 4px 20px rgba(0,0,0,0.08);
    border: 1px solid #e0e7ff;
}

/* Accordion styling */
.accordion {
    background: #f8f9ff;
    border-radius: 12px;
    border: 2px solid #e0e7ff;
}

/* Success/Failure indicators */
.success-badge {
    background: linear-gradient(135deg, #84fab0 0%, #8fd3f4 100%);
    color: #0f5132;
    padding: 8px 16px;
    border-radius: 20px;
    font-weight: 600;
    display: inline-block;
}

.failure-badge {
    background: linear-gradient(135deg, #fa709a 0%, #fee140 100%);
    color: #842029;
    padding: 8px 16px;
    border-radius: 20px;
    font-weight: 600;
    display: inline-block;
}

/* Example buttons */
.example-btn {
    background: #f0f4ff !important;
    border: 2px solid #667eea !important;
    color: #667eea !important;
    border-radius: 8px !important;
    font-weight: 500 !important;
    transition: all 0.2s ease !important;
}

.example-btn:hover {
    background: #667eea !important;
    color: white !important;
}

/* Footer */
.footer {
    text-align: center;
    padding: 30px;
    color: #64748b;
    border-top: 2px solid #e0e7ff;
    margin-top: 40px;
}
"""

# Build interface
with gr.Blocks(css=custom_css, title="Clinical Trial Outcome Predictor", theme=gr.themes.Soft()) as demo:
    
    # header
    gr.HTML("""
    <div class="main-header">
        <h1> Clinical Trial Outcome Predictor</h1>
        <p style="font-size: 20px; margin: 10px 0;">AI-Powered Analysis Using BioBERT + Gemini</p>
        <p style="font-size: 16px; opacity: 0.9;">Trained on 2,000+ clinical trials</p>
    </div>
    """)
    
    prediction_state = gr.State(None)
    
    # Guidance section
    with gr.Accordion(" What Information Should I Provide?", open=True):
        gr.HTML("""
        <div class="info-box">
            <h3 style="color: #667eea; margin-top: 0;">For Best Predictions, Include:</h3>
            <ul style="line-height: 1.8; font-size: 15px;">
                <li>✓ <strong>Study design</strong> (randomized? blinded? controlled?)</li>
                <li>✓ <strong>Patient population</strong> and sample size</li>
                <li>✓ <strong>Intervention details</strong> (drug, dose, schedule)</li>
                <li>✓ <strong>Primary endpoint</strong> (what you're measuring)</li>
                <li>✓ <strong>Trial phase</strong> (Phase 1, 2, or 3)</li>
            </ul>
            <p style="background: #fff3cd; padding: 12px; border-radius: 8px; border-left: 4px solid #ffc107; margin-top: 15px;">
                 <strong>Pro Tip:</strong> Aim for 150+ words in your description for the most accurate prediction!
            </p>
        </div>
        """)
        
        with gr.Accordion(" Example Trial (Click to expand)", open=False):
            gr.Markdown("""
            **Example Title:**  
            `Phase 3 Randomized Study of Nivolumab Plus Ipilimumab in Advanced Melanoma`
            
            **Example Description:**  
            This is a multicenter, randomized, double-blind, Phase 3 study comparing nivolumab plus ipilimumab versus ipilimumab alone in 945 patients with previously untreated, unresectable stage III or IV melanoma. Patients will be randomized 1:1 to receive either nivolumab (1 mg/kg) plus ipilimumab (3 mg/kg) every 3 weeks for 4 doses, followed by nivolumab 3 mg/kg every 2 weeks, or ipilimumab 3 mg/kg every 3 weeks for 4 doses followed by placebo. The primary endpoint is progression-free survival, assessed by blinded independent central review using RECIST 1.1 criteria. Secondary endpoints include overall survival, objective response rate, and safety. The study will be conducted at 137 sites across 21 countries.
            
            **Example Eligibility:**  
            **Inclusion:** Age ≥18 years, histologically confirmed melanoma stage III/IV, ECOG 0-1, measurable disease, no prior systemic therapy.  
            **Exclusion:** Active brain metastases, autoimmune disease, prior anti-CTLA-4/PD-1 therapy, ocular melanoma.
            """)
    
    # Main input/output section
    with gr.Row():
        # Left column - Inputs
        with gr.Column(scale=1):
            gr.HTML('<h2 class="section-header"> Trial Information</h2>')
            
            title_input = gr.Textbox(
                label="Trial Title",
                placeholder="e.g., Phase 3 Study of Drug X in Advanced Melanoma",
                lines=2,
                elem_classes="input-field"
            )
            
            description_input = gr.Textbox(
                label="Trial Description",
                placeholder="""Describe your trial in detail:
- Study design (randomized, blinded, controlled?)
- Patient population (disease, stage, sample size)
- Intervention (drug name, dose, schedule)
- Primary endpoint and timeline
- Study duration and follow-up

The more detail you provide, the better the prediction!""",
                lines=12,
                elem_classes="input-field"
            )
            
            eligibility_input = gr.Textbox(
                label="Eligibility Criteria (Optional but Helpful)",
                placeholder="""Include key criteria:
- Age requirements
- Disease characteristics
- Prior treatment requirements
- Key exclusions""",
                lines=5,
                elem_classes="input-field"
            )
            
            analyze_btn = gr.Button(
                "🚀 Analyze Trial", 
                variant="primary", 
                size="lg",
                elem_classes="primary-button"
            )
            
            gr.HTML("""
            <div style="background: #f0f9ff; padding: 15px; border-radius: 8px; margin-top: 15px; border-left: 4px solid #0284c7;">
                <p style="margin: 0; color: #0c4a6e; font-size: 14px;">
                    <strong> Character count:</strong> Descriptions with 500+ characters typically receive more accurate predictions.
                </p>
            </div>
            """)
        
        # Right column - Results
        with gr.Column(scale=1):
            gr.HTML('<h2 class="section-header"> Prediction Results</h2>')
            
            results_output = gr.Markdown(elem_classes="results-box")
            
            gr.HTML('<h2 class="section-header" style="margin-top: 30px;"> AI Explanation</h2>')
            
            explanation_output = gr.Markdown(elem_classes="results-box")
    
    # Q&A Section
    gr.HTML("""
    <div style="margin: 40px 0 20px 0;">
        <h2 class="section-header">💬 Ask Follow-Up Questions</h2>
        <p style="color: #64748b; font-size: 15px;">Get detailed explanations about the prediction from our AI assistant</p>
    </div>
    """)
    
    with gr.Row():
        question_input = gr.Textbox(
            label="",
            placeholder="Ask anything: Why this prediction? How to improve chances? What are the main concerns?",
            lines=2,
            scale=4,
            elem_classes="input-field"
        )
        ask_btn = gr.Button(
            "💬 Ask AI", 
            variant="secondary", 
            scale=1,
            size="lg"
        )
    
    answer_output = gr.Markdown(elem_classes="results-box")
    
    # Example questions
    gr.HTML('<p style="color: #64748b; font-size: 14px; margin: 20px 0 10px 0;">💡 Quick Questions:</p>')
    
    with gr.Row():
        example_q1 = gr.Button("Why this prediction?", size="sm", elem_classes="example-btn")
        example_q2 = gr.Button("How to improve chances?", size="sm", elem_classes="example-btn")
        example_q3 = gr.Button("What are the risks?", size="sm", elem_classes="example-btn")
        example_q4 = gr.Button("Should we proceed?", size="sm", elem_classes="example-btn")
    
    # Footer
    gr.HTML("""
    <div class="footer">
        <p style="font-size: 14px; margin: 5px 0;">
            <strong style="color: #dc2626;">⚠️ Disclaimer:</strong> This tool is for research and educational purposes only.
        </p>
        <p style="font-size: 13px; color: #94a3b8; margin: 5px 0;">
            Predictions should not be the sole basis for clinical trial decisions. Always consult with clinical trial experts.
        </p>
        <p style="font-size: 13px; margin-top: 15px; color: #64748b;">
            Model: BioBERT fine-tuned on 472 clinical trials | AI Assistant: Gemini 2.5 Flash
        </p>
        <p style="font-size: 12px; color: #94a3b8;">
            Created by Guhan Venkat
        </p>
        <p style="font-size: 12px; color: #94a3b8;">
            MIT License
        </p>
    </div>
    """)
    
    # Connect functions
    analyze_btn.click(
        fn=analyze_trial,
        inputs=[title_input, description_input, eligibility_input],
        outputs=[results_output, explanation_output, prediction_state]
    )
    
    ask_btn.click(
        fn=answer_question,
        inputs=[question_input, title_input, description_input, prediction_state],
        outputs=answer_output
    )
    
    # Example question buttons
    example_q1.click(lambda: "Why did the model make this prediction?", outputs=question_input)
    example_q2.click(lambda: "What could improve the trial's chances of success?", outputs=question_input)
    example_q3.click(lambda: "What are the main risk factors for this trial?", outputs=question_input)
    example_q4.click(lambda: "Based on this prediction, should we proceed with this trial?", outputs=question_input)

# Launch with share link
print("\n Launching Clinical Trial Analyzer...")
print("="*70)
demo.launch(share=True, server_port=None, show_error=True, inbrowser=True)